<a href="https://colab.research.google.com/github/anamitra-tech/ML-Projects/blob/main/EmotionTracker6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 52.8 MB/s eta 0:00:00


In [12]:
import os, re, warnings
warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.impute import SimpleImputer

print("Arvyax - XGBoost (HistGradientBoosting) Pipeline")
print("=" * 55)

# ==============================================================
# DATA
# ==============================================================

train_df = pd.read_csv('/content/Sample_arvyax_reflective_dataset.xlsx - Dataset_120.csv')
test_df  = pd.read_csv('/content/arvyax_test_inputs_120.xlsx - Sheet1.csv')
print(f"train: {len(train_df)}  test: {len(test_df)}")

EMOTIONAL_STATES  = ["calm", "focused", "mixed", "neutral", "overwhelmed", "restless"]
train_df["intensity_3"] = train_df["intensity"].map({
    1:0, 2:0,
    3:1,
    4:2, 5:2
})
INTENSITY_CLASSES = [0,1,2]

# ==============================================================
# FEATURES
# ==============================================================

EMOTION_VOCAB = {
    "calm":        ["calm","settle","settled","quiet","peaceful","lighter","ease","grounded","slow","soft","slowed","soften","serene","centered","breathe","breath","float","release","stillness","pause","relief","less tense"],
    "restless":    ["restless","jumpy","racing","fidgety","scattered","distracted","buzz","switch","bounce","itchy","unable","wander","still busy","mind jumping","low buzz","keep wanting"],
    "focused":     ["focus","focused","clear","plan","organize","prioritize","lock","concentrate","ready","tackle","start","clarity","locked in","sharp","sharper","next steps"],
    "overwhelmed": ["overwhelmed","overloaded","heavy","pressure","carrying","flooded","piled","drained","everything","behind","hard","exhausted","too much","drowning","emotionally tired","want to stop"],
    "neutral":     ["normal","same","steady","average","fine","okay","nothing","fairly","neutral","aware","not much different","mostly same","just normal","no change"],
    "mixed":       ["mixed","split","between","both","part","two","comforted","distracted","uneasy","lingering","conflicted","pulled","still uneasy","better but","not fully","two moods"]
}
MOOD_VOCAB_MAP = {m: set(v) for m, v in EMOTION_VOCAB.items()}
FACE_EMOTIONS  = ["calm_face","happy_face","neutral_face","tired_face","tense_face","none",""]
PREV_MOODS     = ["calm","focused","mixed","neutral","overwhelmed","restless","","none"]
AMBIENCE_CATS  = ["ocean","forest","mountain","rain","cafe"]
TOD_CATS       = ["morning","afternoon","evening","night","early_morning"]


def face_mood_vec(face, prev):
    face = str(face).strip().lower() if pd.notna(face) else "none"
    prev = str(prev).strip().lower() if pd.notna(prev) else ""
    fv = np.zeros(len(FACE_EMOTIONS), dtype=np.float32)
    pv = np.zeros(len(PREV_MOODS),    dtype=np.float32)
    for i,fe in enumerate(FACE_EMOTIONS):
        if fe==face: fv[i]=1.0; break
    for i,pm in enumerate(PREV_MOODS):
        if str(pm).lower()==prev: pv[i]=1.0; break
    return np.concatenate([fv, pv])


def sem_sim_vec(j):
    if not isinstance(j, str): j=""
    tok = set(re.findall(r'\b\w+\b', j.lower()))
    return np.array([len(tok&v)/(np.sqrt(len(tok)+1)*np.sqrt(len(v)+1))
                     for v in MOOD_VOCAB_MAP.values()], dtype=np.float32)


def amb_proximity_vec(j, a):
    if not isinstance(j, str): j=""
    if not isinstance(a, str): a=""
    jl, al  = j.lower(), a.lower()
    tokens  = re.findall(r'\b\w+\b', jl)
    amb_pos = [i for i,t in enumerate(tokens) if t==al]
    scores  = []
    for kws in EMOTION_VOCAB.values():
        s=0.0
        for kw in kws:
            if kw in jl:
                base=1.0
                if amb_pos:
                    kp=[i for i,t in enumerate(tokens) if t==kw.split()[0]]
                    for ap in amb_pos:
                        for k in kp: base=max(base,2.0/(1+abs(ap-k)*0.1))
                s+=base
        scores.append(s)
    tot=sum(scores)+1e-9
    return np.array([s/tot for s in scores]+[float(al in jl)], dtype=np.float32)


def onehot(val, cats):
    v=np.zeros(len(cats), dtype=np.float32)
    s=str(val).lower().strip() if pd.notna(val) else ""
    for i,c in enumerate(cats):
        if c==s: v[i]=1.0; break
    return v


def get_num(row, col, default=3.0):
    val=row.get(col, default)
    try: return float(val) if pd.notna(val) else default
    except: return default


def build_structured(df):
    rows=[]
    for _, row in df.iterrows():
        j=row.get("journal_text",""); a=row.get("ambience_type","")
        sem=sem_sim_vec(j)
        dom=np.zeros(6, dtype=np.float32); dom[np.argmax(sem)]=1.0
        ss=np.sort(sem)[::-1]
        num=np.array([get_num(row,"duration_min",15), get_num(row,"sleep_hours",6),
                      get_num(row,"energy_level",3),  get_num(row,"stress_level",3)], dtype=np.float32)
        rows.append(np.concatenate([
            sem, amb_proximity_vec(j,a),
            face_mood_vec(row.get("face_emotion_hint",""), row.get("previous_day_mood","")),
            onehot(a, AMBIENCE_CATS), onehot(row.get("time_of_day",""), TOD_CATS),
            np.array([{"vague":0.,"conflicted":.5,"clear":1.}.get(
                str(row.get("reflection_quality","vague")).lower().strip(), .25)], dtype=np.float32),
            dom, np.array([ss[0]-ss[1]], dtype=np.float32), num
        ]))
    return np.array(rows, dtype=np.float32)


def make_text(row):
    j=str(row.get("journal_text","")) if pd.notna(row.get("journal_text","")) else ""
    return f"{j} {row.get('ambience_type','')} {row.get('face_emotion_hint','')} {row.get('previous_day_mood','')}"


# ==============================================================
# BUILD FEATURES - tfidf fit on train only
# ==============================================================

print("\n[1/4] Building features...")

train_texts = [make_text(r) for _,r in train_df.iterrows()]
test_texts  = [make_text(r) for _,r in test_df.iterrows()]

tfidf_w = TfidfVectorizer(ngram_range=(1,2), max_features=500, sublinear_tf=True, min_df=3)
tfidf_c = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,4), max_features=200,
                           sublinear_tf=True, min_df=4)

Tw_tr = tfidf_w.fit_transform(train_texts).toarray()
Tc_tr = tfidf_c.fit_transform(train_texts).toarray()
Tw_te = tfidf_w.transform(test_texts).toarray()
Tc_te = tfidf_c.transform(test_texts).toarray()

svd_w = TruncatedSVD(min(40, Tw_tr.shape[1]-1), random_state=42)
svd_c = TruncatedSVD(min(20, Tc_tr.shape[1]-1), random_state=42)
Tw_tr = svd_w.fit_transform(Tw_tr); Tw_te = svd_w.transform(Tw_te)
Tc_tr = svd_c.fit_transform(Tc_tr); Tc_te = svd_c.transform(Tc_te)

X_s_tr = build_structured(train_df)
X_s_te = build_structured(test_df)

X_tr_raw = np.hstack([Tw_tr, Tc_tr, X_s_tr])
X_te_raw = np.hstack([Tw_te, Tc_te, X_s_te])

imp    = SimpleImputer(strategy='median')
scaler = StandardScaler()
X_tr   = scaler.fit_transform(imp.fit_transform(X_tr_raw))
X_te   = scaler.transform(imp.transform(X_te_raw))

le_cls = LabelEncoder(); le_cls.classes_ = np.array(EMOTIONAL_STATES)
y_cls  = le_cls.transform(train_df["emotional_state"].str.lower().str.strip())
y_int  = train_df["intensity_3"].values

print(f"  features : {X_tr.shape}")

Xc_tr, Xc_val, yi_tr, yi_val, yc_tr, yc_val = train_test_split(
    X_tr, y_int, y_cls, test_size=0.15, random_state=42, stratify=y_cls
)

# ==============================================================
# MODELS
# ==============================================================

print("\n[2/4] Training...")

# Y1 - emotional state
cls_model = HistGradientBoostingClassifier(
    max_iter=300, max_depth=4, learning_rate=0.05,
    min_samples_leaf=10, random_state=42,
    class_weight='balanced'
)
cls_model.fit(Xc_tr, yc_tr)
val_acc = cls_model.score(Xc_val, yc_val)
print(f"  Y1 val accuracy : {val_acc:.3f}")
print(train_df["intensity"].value_counts().sort_index())

# Y2 - intensity as classification
int_model = HistGradientBoostingClassifier(
    max_iter=300, max_depth=3, learning_rate=0.05,
    min_samples_leaf=15, random_state=42
)
int_model.fit(Xc_tr, yi_tr)
val_int = int_model.score(Xc_val, yi_val)
print(f"  Y2 val accuracy : {val_int:.3f}  (5-class, random baseline = 0.20)")

# ==============================================================
# EVALUATION
# ==============================================================

print("\n[3/4] Evaluating on validation set...")

yc_pred = cls_model.predict(Xc_val)
print(classification_report(yc_val, yc_pred,
      target_names=EMOTIONAL_STATES, zero_division=0))

yi_pred = int_model.predict(Xc_val)
print("Classification Report - Intensity (Y2)")
print(classification_report(yi_val, yi_pred,
      target_names=["low","medium","high"]))
# ==============================================================
# ATTENTION-BASED RECOMMENDATION (no if/else)
# ==============================================================

TEMPLATES = [
    {"label":"deep_work",          "kw":["focused","calm","organized","clear"],
     "fn": lambda a,t,d,s: f"Your mind is in a receptive state. Use this for deep work. The {a} ambience supported focus today. Start with the hardest task while this clarity holds."},
    {"label":"gentle_reset",       "kw":["calm","settled","lighter","peaceful"],
     "fn": lambda a,t,d,s: f"You've settled into a quieter headspace. The {a} ambience anchored this. A short pause or light movement will carry it further into {t}."},
    {"label":"grounding_practice", "kw":["restless","jumpy","scattered","racing"],
     "fn": lambda a,t,d,s: f"Your system is still running fast. The {a} sounds can anchor you - sync your breath slowly. One task at a time will bring the buzz down."},
    {"label":"emotional_offload",  "kw":["overwhelmed","heavy","flooded","pressure"],
     "fn": lambda a,t,d,s: f"You're carrying a lot right now. The {a} ambience has been doing quiet work. Try a journal dump or short walk before returning to demands."},
    {"label":"dual_awareness",     "kw":["mixed","split","between","uneasy"],
     "fn": lambda a,t,d,s: f"Two emotional currents are running. The {a} setting softened the gap. Don't force resolution - one anchor task will pull you forward."},
    {"label":"steady_continuity",  "kw":["neutral","steady","same","fine"],
     "fn": lambda a,t,d,s: f"Your baseline is stable. The {a} session kept things even. Good state for routine work or small creative steps during {t}."},
    {"label":"rest_recovery",      "kw":["tired","tired_face","drained","exhausted"],
     "fn": lambda a,t,d,s: f"Fatigue is present. The {a} soundscape offered some softening. Sleep and genuine stillness are the highest-return action right now."},
    {"label":"high_intensity",     "kw":["tense_face","tense","wound","unable"],
     "fn": lambda a,t,d,s: f"Physical tension is elevated. Use {a} as a reset between tasks. Break obligations into smaller steps to reduce the felt pressure."},
]

KEY_DIM = len(EMOTIONAL_STATES) + 1

def build_key(kws):
    key=np.zeros(KEY_DIM, dtype=np.float32)
    sm={s:i for i,s in enumerate(EMOTIONAL_STATES)}
    for kw in kws:
        for state,idx in sm.items():
            if kw in state or state in kw or kw in MOOD_VOCAB_MAP.get(state,set()):
                key[idx]+=1.0
        if kw in ["tired","tense","tense_face","exhausted","drained"]: key[-1]+=1.0
    return key/(np.linalg.norm(key)+1e-9)

KEY_MTX = np.array([build_key(t["kw"]) for t in TEMPLATES])


def attention_recommend(cls_probs, intensity_bin, ambience, time_of_day, duration_min, sleep_hours):
    # intensity_bin is 0-indexed (0-4), normalize to 0-1
    Q    = np.append(cls_probs, intensity_bin/4.0).astype(np.float32)
    attn = np.exp(KEY_MTX @ Q / np.sqrt(KEY_DIM)); attn /= attn.sum()
    tmpl = TEMPLATES[int(np.argmax(attn))]
    return {
        "recommendation":          tmpl["fn"](ambience, time_of_day, duration_min, sleep_hours),
        "top_template":            tmpl["label"],
        "duration_min":            int(duration_min),
        "sleep_hours_recommended": 8 if sleep_hours < 6 else round(sleep_hours, 1),
        "time_of_day":             time_of_day,
        "attention_weights":       {t["label"]:float(w) for t,w in zip(TEMPLATES,attn)},
    }


# ==============================================================
# INFERENCE ON TEST DATA
# ==============================================================

print("\n[4/4] Inference on test data...")

cls_probs_test = cls_model.predict_proba(X_te)
cls_pred_test  = np.argmax(cls_probs_test, axis=1)
int_pred_test  = int_model.predict(X_te)   # 0-indexed

lbl_test = le_cls.classes_[cls_pred_test]
int_lbl  = int_pred_test + 1               # back to 1-5

results=[]
for i, row in test_df.iterrows():
    idx = i - test_df.index[0]
    amb = str(row.get("ambience_type","")).lower()
    tod = str(row.get("time_of_day","")).lower()
    dur = float(row.get("duration_min",10))
    sl  = float(row.get("sleep_hours",7)) if pd.notna(row.get("sleep_hours")) else 7.0
    rec = attention_recommend(cls_probs_test[idx], int_pred_test[idx], amb, tod, dur, sl)
    results.append({
        "id":                        row["id"],
        "predicted_emotional_state": lbl_test[idx],
        "predicted_intensity":       int(int_lbl[idx]),
        "recommendation":            rec["recommendation"],
        "top_template":              rec["top_template"],
        "duration_min":              rec["duration_min"],
        "sleep_hours_recommended":   rec["sleep_hours_recommended"],
        "time_of_day":               rec["time_of_day"],
        "attention_weights":         rec["attention_weights"],
        "cls_confidence":            round(float(cls_probs_test[idx].max()), 3),
        "ambience_type":             row.get("ambience_type",""),
    })




Arvyax - XGBoost (HistGradientBoosting) Pipeline
train: 1200  test: 120

[1/4] Building features...
  features : (1200, 110)

[2/4] Training...
  Y1 val accuracy : 0.522
intensity
1    226
2    228
3    240
4    277
5    229
Name: count, dtype: int64
  Y2 val accuracy : 0.422  (5-class, random baseline = 0.20)

[3/4] Evaluating on validation set...
              precision    recall  f1-score   support

        calm       0.61      0.59      0.60        32
     focused       0.59      0.55      0.57        29
       mixed       0.57      0.55      0.56        29
     neutral       0.50      0.47      0.48        30
 overwhelmed       0.43      0.41      0.42        29
    restless       0.45      0.55      0.49        31

    accuracy                           0.52       180
   macro avg       0.53      0.52      0.52       180
weighted avg       0.53      0.52      0.52       180

Classification Report - Intensity (Y2)
              precision    recall  f1-score   support

         low